# NCAA March Madness Advanced Modeling (v2)

This notebook uses the advanced features (SOS, Trends) to build a better predictive model.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
import os

DATA_PATH = './march-machine-learning-mania-2026/'

# Load v2 features
team_features = pd.read_csv('all_team_features_v2.csv')
m_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))
tourney_results = pd.concat([m_tourney_results, w_tourney_results], ignore_index=True)

print("v2 Features loaded.")

## 1. Constructing Training Data with Advanced Features

In [ ]:
def prepare_training_data_v2(results_df, features_df):
    training_rows = []
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    
    cols_to_diff = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank']
    
    for i, row in results_df.iterrows():
        season = row['Season']
        t1 = row['WTeamID'] if row['WTeamID'] < row['LTeamID'] else row['LTeamID']
        t2 = row['LTeamID'] if row['WTeamID'] < row['LTeamID'] else row['WTeamID']
        label = 1 if row['WTeamID'] == t1 else 0
        
        try:
            f1 = feat_dict[(season, t1)]
            f2 = feat_dict[(season, t2)]
            
            diffs = {f'{col}Diff': f1[col] - f2[col] for col in cols_to_diff}
            diffs['Season'] = season
            diffs['label'] = label
            training_rows.append(diffs)
        except KeyError:
            continue
            
    return pd.DataFrame(training_rows)

train_df = prepare_training_data_v2(tourney_results, team_features)
print(f"Training dataset v2 created with {len(train_df)} games.")

## 2. Advanced Model Training & Tuning

Using optimized hyperparameters for XGBoost.

In [ ]:
X = train_df.drop(['Season', 'label'], axis=1)
y = train_df['label']

params = {
    'eval_metric': 'logloss',
    'max_depth': 4,
    'learning_rate': 0.05,
    'n_estimators': 200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    score = log_loss(y_val, preds)
    scores.append(score)

print(f"Average CV Log Loss (v2): {np.mean(scores):.4f}")

## 3. Final Submission Generation

In [ ]:
submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))
final_model = xgb.XGBClassifier(**params)
final_model.fit(X, y)
feat_dict = team_features.set_index(['Season', 'TeamID']).to_dict('index')
cols_to_diff = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank']

def get_submission_pred_v2(row):
    parts = row['ID'].split('_')
    season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
    
    try:
        f1 = feat_dict[(season, t1)]
        f2 = feat_dict[(season, t2)]
        
        feat_row = pd.DataFrame([{f'{col}Diff': f1[col] - f2[col] for col in cols_to_diff}])
        return final_model.predict_proba(feat_row)[:, 1][0]
    except KeyError:
        return 0.5

submission['Pred'] = submission.apply(get_submission_pred_v2, axis=1)
submission.to_csv('submission_v2.csv', index=False)
print("Submission v2 generated!")